EfficientSAM Tests

Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision.transforms import ToTensor
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Download COCO

In [ ]:
import fiftyone.zoo as foz
dataset = foz.load_zoo_dataset(
    "coco-2017", split="validation", label_types=["segmentations"],
    max_samples=100, shuffle=True, seed=42, label_field="ground_truth"
)

Running the model

In [ ]:
def run_model(image_np, pts_sampled, pts_labels, model):
    img_tensor = ToTensor()(image_np).to(device)
    pts_sampled = torch.reshape(
        torch.tensor(pts_sampled, device=device),
        [1, 1, -1, 2]
    )
    pts_labels = torch.reshape(
        torch.tensor(pts_labels, device=device),
        [1, 1, -1]
    )
    predicted_logits, predicted_iou = model(
        img_tensor[None, ...],
        pts_sampled,
        pts_labels,
    )

    sorted_ids = torch.argsort(predicted_iou, dim=-1, descending=True)
    predicted_iou = torch.take_along_dim(predicted_iou, sorted_ids, dim=2)
    predicted_logits = torch.take_along_dim(
        predicted_logits, sorted_ids[..., None, None], dim=2
    )

    return torch.ge(predicted_logits[0, 0, 0, :, :], 0).cpu().detach().numpy()

EfficientSAM

In [ ]:
import os

if os.path.basename(os.getcwd()) != "EfficientSAM":
    os.chdir("EfficientSAM")

In [ ]:
from efficient_sam.build_efficient_sam import (
    build_efficient_sam_vitt, build_efficient_sam_vits
)
import zipfile

efficient_sam_vitt_model = build_efficient_sam_vitt().to(device)
efficient_sam_vitt_model.eval()

with zipfile.ZipFile("weights/efficient_sam_vits.pt.zip", "r") as zip_ref:
    zip_ref.extractall("weights")
efficient_sam_vits_model = build_efficient_sam_vits().to(device)
efficient_sam_vits_model.eval()

Helpers

In [ ]:
def get_box_pixels(finding, width, height):
    x, y, w, z = finding.bounding_box

    x1 = int(x * width)
    y1 = int(y * height)
    x2 = int((x + w) * width)
    y2 = int((y + z) * height)

    x1 = max(0, min(x1, width - 1))
    y1 = max(0, min(y1, height - 1))
    x2 = max(x1 + 1, min(x2, width))
    y2 = max(y1 + 1, min(y2, height))
    return x1, y1, x2, y2

def calculate_iou(prediction, mask):
    intersection = np.logical_and(prediction, mask).sum()
    union = np.logical_or(prediction, mask).sum()
    return 1.0 if union == 0 else intersection / union

In [ ]:
def make_gt(detection, width, height):
    x1, y1, x2, y2 = get_box_pixels(detection, width, height)
    mask = detection.mask

    if mask is None:
        return None

    resized_mask = Image.fromarray(mask.astype(np.uint8) * 255).resize((x2 - x1, y2 - y1), Image.Resampling.NEAREST)
    resized_mask = np.array(resized_mask, dtype=np.uint8) > 0

    output_mask = np.zeros((height, width), dtype=bool)
    output_mask[y1:y2, x1:x2] = resized_mask
    return output_mask

Load SAM

In [ ]:
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_h"](checkpoint="checkpoints/sam_vit_h_4b8939.pth").to(device)
sam.eval()
sam_predictor = SamPredictor(sam)

## Run box prompt experiment

In [ ]:
results = []

for i, sample in enumerate(dataset, start=1):
    print(f"Processing image {i}/{len(dataset)}")
    image_path = sample.filepath
    image = np.array(Image.open(image_path).convert("RGB"))
    height, width = image.shape[:2]

    if sample.ground_truth is None:
        continue

    sam_predictor.set_image(image)
    for det_index, det in enumerate(sample.ground_truth.detections):
        if det.mask is None:
            continue

        gt_mask = make_gt(det, width, height)
        x1, y1, x2, y2 = get_box_pixels(det, width, height)
        box = [x1, y1, x2, y2]
        box_points = np.array([[x1, y1], [x2, y2]])
        box_labels = np.array([2, 3])

        with torch.no_grad():
            mask_ti = run_model(image, box_points, box_labels, efficient_sam_vitt_model)
            mask_s = run_model(image, box_points, box_labels, efficient_sam_vits_model)
            masks, scores, _ = sam_predictor.predict(box=np.array(box), multimask_output=True)
            mask_sam = masks[np.argmax(scores)]

        results.append({
            "image": image_path,
            "det_index": det_index,
            "class": det.label,
            "box": box,
            "iou_ti": calculate_iou(mask_ti, gt_mask),
            "iou_s": calculate_iou(mask_s, gt_mask),
            "iou_sam": calculate_iou(mask_sam, gt_mask)
        })

## Point Prompt Experiment

In [ ]:
point_rng = np.random.default_rng(42)


def get_point_prompt(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    selected = point_rng.integers(len(xs))
    return np.array([[xs[selected], ys[selected]]])

In [ ]:
point_results = []

for i, sample in enumerate(dataset, start=1):
    print(f"Processing image {i}/{len(dataset)}")
    image_path = sample.filepath
    image = np.array(Image.open(image_path).convert("RGB"))
    height, width = image.shape[:2]

    if sample.ground_truth is None:
        continue

    sam_predictor.set_image(image)
    for det_index, det in enumerate(sample.ground_truth.detections):
        if det.mask is None:
            continue

        gt_mask = make_gt(det, width, height)
        point = get_point_prompt(gt_mask)
        if point is None:
            continue

        labels = np.array([1])
        with torch.no_grad():
            mask_ti = run_model(image, point, labels, efficient_sam_vitt_model)
            mask_s = run_model(image, point, labels, efficient_sam_vits_model)
            masks, scores, _ = sam_predictor.predict(point_coords=point, point_labels=labels, multimask_output=True)
            mask_sam = masks[np.argmax(scores)]

        point_results.append({
            "image": image_path,
            "det_index": det_index,
            "class": det.label,
            "point": point.tolist(),
            "iou_ti": calculate_iou(mask_ti, gt_mask),
            "iou_s": calculate_iou(mask_s, gt_mask),
            "iou_sam": calculate_iou(mask_sam, gt_mask)
        })

## Mean IoU Comparison

In [ ]:
model_names = ["EfficientSAM-Ti", "EfficientSAM-S", "SAM"]
box_means = [
    np.mean([r["iou_ti"] for r in results]),
    np.mean([r["iou_s"] for r in results]),
    np.mean([r["iou_sam"] for r in results])
]
point_means = [
    np.mean([r["iou_ti"] for r in point_results]),
    np.mean([r["iou_s"] for r in point_results]),
    np.mean([r["iou_sam"] for r in point_results])
]

x = np.arange(len(model_names))
w = 0.35
plt.figure(figsize=(10, 6))
plt.bar(x - w / 2, box_means, w, label="Box Prompt")
plt.bar(x + w / 2, point_means, w, label="Point Prompt")
plt.xticks(x, model_names)
plt.ylabel("Mean IoU")
plt.title("Mean IoU by Model and Prompt Type")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

## Gaussian Noise Experiment

In [ ]:
def add_gaussian_noise(image, sigma, rng):
    if sigma == 0:
        return image.copy()

    noise = rng.normal(0, sigma, image.shape)
    noisy = image.astype(np.float32) + noise

    return np.clip(noisy, 0, 255).astype(np.uint8)

In [ ]:
noise_levels = [0, 10, 20, 30, 40, 50]
noise_results = []
rng = np.random.default_rng(42)

for i, sample in enumerate(dataset, start=1):
    print(f"Processing image {i}/{len(dataset)}")
    image_path = sample.filepath
    image = np.array(Image.open(image_path).convert("RGB"))
    height, width = image.shape[:2]

    if sample.ground_truth is None:
        continue

    for sigma in noise_levels:
        noisy_image = add_gaussian_noise(image, sigma, rng)
        sam_predictor.set_image(noisy_image)

        for det in sample.ground_truth.detections:
            if det.mask is None:
                continue

            gt_mask = make_gt(det, width, height)
            x1, y1, x2, y2 = get_box_pixels(det, width, height)
            box = [x1, y1, x2, y2]
            box_points = np.array([[x1, y1], [x2, y2]])
            box_labels = np.array([2, 3])

            with torch.no_grad():
                mask_ti = run_model(noisy_image, box_points, box_labels, efficient_sam_vitt_model)
                mask_s = run_model(noisy_image, box_points, box_labels, efficient_sam_vits_model)
                masks, scores, _ = sam_predictor.predict(box=np.array(box), multimask_output=True)
                mask_sam = masks[np.argmax(scores)]

            noise_results.append({
                "noise": sigma,
                "iou_ti": calculate_iou(mask_ti, gt_mask),
                "iou_s": calculate_iou(mask_s, gt_mask),
                "iou_sam": calculate_iou(mask_sam, gt_mask)
            })

## Gaussian Noise Results

In [ ]:
noise_means = {}

for sigma in noise_levels:
    current = [r for r in noise_results if r["noise"] == sigma]
    noise_means[sigma] = {
        "ti": np.mean([r["iou_ti"] for r in current]),
        "s": np.mean([r["iou_s"] for r in current]),
        "sam": np.mean([r["iou_sam"] for r in current])
    }

plt.figure(figsize=(8, 5))
plt.plot(noise_levels, [noise_means[n]["ti"] for n in noise_levels], marker="o", label="EfficientSAM-Ti")
plt.plot(noise_levels, [noise_means[n]["s"] for n in noise_levels], marker="o", label="EfficientSAM-S")
plt.plot(noise_levels, [noise_means[n]["sam"] for n in noise_levels], marker="o", label="SAM")
plt.xlabel("Gaussian Noise")
plt.ylabel("Mean IoU")
plt.title("Mean IoU vs Gaussian Noise")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Shifted Box Prompt Experiment

In [ ]:
def shifted_box_prompt(det, width, height, shift):
    x1, y1, x2, y2 = get_box_pixels(det, width, height)
    box_width, box_height = x2 - x1, y2 - y1
    dx, dy = int(box_width * shift), int(box_height * shift)

    new_x1 = max(0, min(x1 + dx, width - box_width))
    new_y1 = max(0, min(y1 + dy, height - box_height))
    return new_x1, new_y1, new_x1 + box_width, new_y1 + box_height

In [ ]:
shift_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
shift_results = []

for i, sample in enumerate(dataset, start=1):
    print(f"Processing image {i}/{len(dataset)}")
    image_path = sample.filepath
    image = np.array(Image.open(image_path).convert("RGB"))
    height, width = image.shape[:2]

    if sample.ground_truth is None:
        continue

    sam_predictor.set_image(image)

    for det in sample.ground_truth.detections:
        if det.mask is None:
            continue

        gt_mask = make_gt(det, width, height)

        for shift in shift_levels:
            x1, y1, x2, y2 = shifted_box_prompt(det, width, height, shift)
            box = [x1, y1, x2, y2]
            box_points = np.array([[x1, y1], [x2, y2]])
            box_labels = np.array([2, 3])

            with torch.no_grad():
                mask_ti = run_model(image, box_points, box_labels, efficient_sam_vitt_model)
                mask_s = run_model(image, box_points, box_labels, efficient_sam_vits_model)
                masks, scores, _ = sam_predictor.predict(box=np.array(box), multimask_output=True)
                mask_sam = masks[np.argmax(scores)]

            shift_results.append({
                "shift": shift * 100,
                "iou_ti": calculate_iou(mask_ti, gt_mask),
                "iou_s": calculate_iou(mask_s, gt_mask),
                "iou_sam": calculate_iou(mask_sam, gt_mask)
            })

## Shifted Box Prompt Results

In [ ]:
shift_percentages = [shift * 100 for shift in shift_levels]
shift_means = {}

for shift in shift_percentages:
    current = [r for r in shift_results if r["shift"] == shift]
    shift_means[shift] = {
        "ti": np.mean([r["iou_ti"] for r in current]),
        "s": np.mean([r["iou_s"] for r in current]),
        "sam": np.mean([r["iou_sam"] for r in current])
    }

plt.figure(figsize=(8, 5))
plt.plot(shift_percentages, [shift_means[n]["ti"] for n in shift_percentages], marker="o", label="EfficientSAM-Ti")
plt.plot(shift_percentages, [shift_means[n]["s"] for n in shift_percentages], marker="o", label="EfficientSAM-S")
plt.plot(shift_percentages, [shift_means[n]["sam"] for n in shift_percentages], marker="o", label="SAM")
plt.xlabel("Box Shift (%)")
plt.ylabel("Mean IoU")
plt.title("Mean IoU vs Box Prompt Shift")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Best and Worst Examples


In [ ]:
def show_prediction(image, mask, ax):
    ax.imshow(image)
    ax.imshow(mask, alpha=0.45)
    ax.axis("off")

In [ ]:
def show_box(box, ax):
    x1, y1, x2, y2 = box
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,fill=False, edgecolor="red", linewidth=2))

## Best and Worst Box Prompt Examples

In [ ]:
models = {
    "EfficientSAM-Ti": ("iou_ti", efficient_sam_vitt_model),
    "EfficientSAM-S": ("iou_s", efficient_sam_vits_model),
    "SAM": ("iou_sam", None)
}

fig, axes = plt.subplots(3, 2, figsize=(12, 15))

for row, (model_name, (iou_key, model)) in enumerate(models.items()):
    best = max(results, key=lambda x: x[iou_key])
    worst = min(results, key=lambda x: x[iou_key])

    for col, (name, example) in enumerate([("Best", best), ("Worst", worst)]):
        image = np.array(Image.open(example["image"]).convert("RGB"))
        x1, y1, x2, y2 = example["box"]
        box_points = np.array([[x1, y1], [x2, y2]])
        box_labels = np.array([2, 3])

        if model_name == "SAM":
            sam_predictor.set_image(image)
            masks, scores, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]), multimask_output=True)
            prediction = masks[np.argmax(scores)]
        else:
            prediction = run_model(image, box_points, box_labels, model)

        ax = axes[row, col]
        show_prediction(image, prediction, ax)
        show_box([x1, y1, x2, y2], ax)
        ax.set_title(model_name)

plt.tight_layout()
plt.show()

## Best and Worst Point Prompt Examples

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 15))

for row, (model_name, (iou_key, model)) in enumerate(models.items()):
    best = max(point_results, key=lambda x: x[iou_key])
    worst = min(point_results, key=lambda x: x[iou_key])

    for col, (name, example) in enumerate([("Best", best), ("Worst", worst)]):
        image = np.array(Image.open(example["image"]).convert("RGB"))
        point = np.array(example["point"])
        labels = np.array([1])

        if model_name == "SAM":
            sam_predictor.set_image(image)
            masks, scores, _ = sam_predictor.predict(point_coords=point, point_labels=labels, multimask_output=True)
            prediction = masks[np.argmax(scores)]
        else:
            prediction = run_model(image, point, labels, model)

        ax = axes[row, col]
        show_prediction(image, prediction, ax)
        ax.scatter(point[:, 0], point[:, 1], marker="*", s=250, edgecolor="white", linewidth=1.5)
        ax.set_title(model_name)

plt.tight_layout()
plt.show()

## Timing Experiment

In [ ]:
import time

timing_results = []

def synchronize_cuda():
    if device.type == "cuda":
        torch.cuda.synchronize()

for sample in dataset:
    if len(timing_results) >= 5:
        break

    image = np.array(Image.open(sample.filepath).convert("RGB"))
    height, width = image.shape[:2]
    if sample.ground_truth is None:
        continue

    for det in sample.ground_truth.detections:
        if det.mask is None:
            continue

        x1, y1, x2, y2 = get_box_pixels(det, width, height)
        box = [x1, y1, x2, y2]
        box_points = np.array([[x1, y1], [x2, y2]])
        box_labels = np.array([2, 3])

        with torch.no_grad():
            synchronize_cuda()
            start = time.perf_counter()
            run_model(image, box_points, box_labels, efficient_sam_vitt_model)
            synchronize_cuda()
            ti_time = time.perf_counter() - start

            synchronize_cuda()
            start = time.perf_counter()
            run_model(image, box_points, box_labels, efficient_sam_vits_model)
            synchronize_cuda()
            s_time = time.perf_counter() - start

            synchronize_cuda()
            start = time.perf_counter()
            sam_predictor.set_image(image)
            masks, scores, _ = sam_predictor.predict(box=np.array(box), multimask_output=True)
            _ = masks[np.argmax(scores)]
            synchronize_cuda()
            sam_time = time.perf_counter() - start

        timing_results.append({
            "image": sample.filepath,
            "efficient_sam_ti_ms": ti_time * 1000,
            "efficient_sam_s_ms": s_time * 1000,
            "sam_ms": sam_time * 1000
        })
        break

model_names = ["EfficientSAM-Ti", "EfficientSAM-S", "SAM"]
average_times = [
    np.mean([r["efficient_sam_ti_ms"] for r in timing_results]),
    np.mean([r["efficient_sam_s_ms"] for r in timing_results]),
    np.mean([r["sam_ms"] for r in timing_results])
]

plt.figure(figsize=(8, 5))
plt.bar(model_names, average_times)
plt.ylabel("Average Time (ms)")
plt.title("Average Time")
plt.tight_layout()
plt.show()